# 🚀 Stack 2.9 - Colab Training Notebook

**Zero-cost training on Google Colab free tier with T4 GPU**

⏱️ **Expected runtime:** 3-5 hours
💾 **VRAM needed:** ~12GB (fits in T4's 15GB)

---

**CRITICAL:** Run cells in order from the top!

---

In [ ]:
# STEP 1: Setup - Mount Drive and define root directory
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT_DIR = "/content/drive/MyDrive/stack-2.9"
os.makedirs(ROOT_DIR, exist_ok=True)
os.chdir(ROOT_DIR)

# Define all paths once
REPO_DIR = os.path.join(ROOT_DIR, "stack-2.9")
MODEL_DIR = os.path.join(REPO_DIR, "base_model_qwen7b")
OUTPUT_DIR = os.path.join(ROOT_DIR, "training_output")

print(f"✅ ROOT_DIR: {ROOT_DIR}")
print(f"✅ REPO_DIR: {REPO_DIR}")
print(f"✅ MODEL_DIR: {MODEL_DIR}")
print(f"✅ OUTPUT_DIR: {OUTPUT_DIR}")
!ls -la

In [ ]:
# STEP 2: Clone repo (with retry logic)
import shutil
import time

max_retries = 3
for attempt in range(max_retries):
    try:
        if os.path.exists('stack-2.9'):
            print(f"Attempt {attempt+1}: Removing old stack-2.9...")
            shutil.rmtree('stack-2.9')
        
        print(f"Attempt {attempt+1}: Cloning repository...")
        result = !git clone https://github.com/my-ai-stack/stack-2.9.git
        
        if os.path.exists('stack-2.9'):
            print("✅ Clone successful!")
            break
    except Exception as e:
        print(f"⚠️ Attempt {attempt+1} failed: {e}")
        if attempt < max_retries - 1:
            print("Retrying in 5 seconds...")
            time.sleep(5)
else:
    raise RuntimeError("Failed to clone repository after 3 attempts")

os.chdir(REPO_DIR)
print(f"✅ In: {os.getcwd()}")
!ls -la

In [ ]:
# STEP 3: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers peft accelerate datasets pyyaml tqdm scipy bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# STEP 4: Download Base Model (Qwen2.5-Coder-7B)
# Check if model already exists FIRST before trying to download
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B"

# Check if model files already exist (don't try to download first)
if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print(f"✅ Model already exists at: {MODEL_DIR}")
elif os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")):
    print(f"✅ Model partially exists, verifying...")
    # Just verify, don't download
else:
    print(f"⚠️ Model not found at: {MODEL_DIR}")
    print("⏭️ SKIPPING model download to avoid crash...")
    print("   To train, you'll need to:")
    print("   1. Download model locally using Ollama")
    print("   2. Upload model files to Drive manually")
    print("   OR use a smaller model")

# Continue even without model - training step will handle it
print(f"\nModel dir check: {os.path.exists(MODEL_DIR)}")
if os.path.exists(MODEL_DIR):
    !ls -lh {MODEL_DIR} | head -5

In [ ]:
# STEP 5: Find or download training data
import json

DATA_PATH = None

# Check multiple possible locations
possible_paths = [
    os.path.join(REPO_DIR, "data/final/train.jsonl"),
    os.path.join(REPO_DIR, "training-data/final/train.jsonl"),
    os.path.join(REPO_DIR, "data_mini/train_mini.jsonl"),
]

for path in possible_paths:
    if os.path.exists(path):
        DATA_PATH = path
        print(f"✅ Found data at: {path}")
        break

In [ ]:
# STEP 6: Prepare Training Configuration
import yaml

config_path = os.path.join(REPO_DIR, "stack/training/train_config_local.yaml")

if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at: {config_path}")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update config with absolute paths
config['model']['name'] = MODEL_DIR
config['data']['input_path'] = DATA_PATH
config['output']['lora_dir'] = os.path.join(OUTPUT_DIR, "lora")
config['output']['merged_dir'] = os.path.join(OUTPUT_DIR, "merged")
config['hardware']['device'] = "cuda"
config['hardware']['num_gpus'] = 1

os.makedirs(OUTPUT_DIR, exist_ok=True)
updated_config_path = os.path.join(OUTPUT_DIR, "train_config.yaml")

with open(updated_config_path, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config saved to: {updated_config_path}")
print(f"   Model: {config['model']['name']}")
print(f"   Data: {config['data']['input_path']}")

In [ ]:
# STEP 7: Train LoRA Adapter
import os
import sys

# Check if model exists before training
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("❌ Model not found! Cannot train without base model.")
    print(f"Expected at: {MODEL_DIR}")
    raise RuntimeError("Model missing - please upload base model to Drive first")

sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))

print("="*60)
print("STARTING TRAINING")
print("="*60)

from train_lora import train_lora
trainer = train_lora(updated_config_path)

print("="*60)
print("TRAINING COMPLETED")
print("="*60)

In [ ]:
# STEP 8: Verify and Merge
lora_dir = os.path.join(OUTPUT_DIR, "lora")
print(f"Checking LoRA: {lora_dir}")
if os.path.exists(lora_dir):
    !ls -lh {lora_dir}
else:
    print("❌ No LoRA output found")

In [ ]:
# STEP 9: Merge LoRA
import os  # Make sure os is imported
import yaml
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))
from merge_adapter import merge_adapter

## 🔚 Training Complete!

Your model is ready at:
`/content/drive/MyDrive/stack-2.9/training_output/merged/`

Download it from Google Drive!